# 01. Sales Exploratory Data Analysis & Baseline Model
**Author:** Person 1 (Sales Data & Baseline Model)  
**Project:** Retail Demand Forecasting  

This notebook covers two parts:
1. **Exploratory Data Analysis (EDA):** Visualizing sales over time, day of week patterns, promotional effects, and city comparisons.
2. **Baseline Model:** Building a sales-only Linear Regression model using past sales (7-day lag), day of week, and promo flags.

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Set plot style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Resolve paths whether run from workspace root or notebooks/
data_path = "data/processed/sales_clean.csv" if os.path.exists("data/processed/sales_clean.csv") else "../data/processed/sales_clean.csv"
fig_dir = "results/figures" if os.path.exists("results/figures") else "../results/figures"
results_dir = "results" if os.path.exists("results") else "../results"
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## 1. Load Data
Load `data/processed/sales_clean.csv` and inspect basic info.

In [ ]:
# Load cleaned sales data
df = pd.read_csv(data_path, parse_dates=["date"])

print("Shape:", df.shape)
print("Stores count:", df["store_id"].nunique())
print("Date range:", df["date"].min().strftime("%Y-%m-%d"), "to", df["date"].max().strftime("%Y-%m-%d"))
df.head()

## 2. Chart 1: Total Sales Over Time
Aggregate sales across all 50 stores by date to observe demand trends and seasonality over the 2.5-year period.

In [ ]:
# Compute total daily sales across all 50 stores
daily_sales = df.groupby("date")["sales"].sum()

plt.figure(figsize=(12, 5))
plt.plot(daily_sales.index, daily_sales.values, color="#1f77b4", linewidth=1)
plt.title("Total Daily Sales Over Time (All 50 Stores)", fontsize=14, pad=12)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Daily Sales", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# Save chart
plt.savefig(os.path.join(fig_dir, "sales_over_time.png"), dpi=150)
plt.show()

## 3. Chart 2: Average Sales by Day of Week
Compare average store sales across days of the week. Stores in Germany are closed on Sundays (`Open == 0`).

In [ ]:
# Calculate average sales for each day of the week
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_name"] = df["date"].dt.day_name()
avg_dow = df.groupby("day_name")["sales"].mean().reindex(days_order).fillna(0)

plt.figure(figsize=(9, 5))
bars = plt.bar(avg_dow.index, avg_dow.values, color="#2ca02c", edgecolor="black", alpha=0.85)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, height + 100, f"{int(height):,}", ha="center", va="bottom", fontsize=9)
    else:
        plt.text(bar.get_x() + bar.get_width() / 2, 100, "Closed", ha="center", va="bottom", fontsize=9, color="red")

plt.title("Average Daily Sales by Day of Week", fontsize=14, pad=12)
plt.xlabel("Day of Week", fontsize=12)
plt.ylabel("Average Sales per Store", fontsize=12)
plt.ylim(0, 9000)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

# Save chart
plt.savefig(os.path.join(fig_dir, "sales_by_day_of_week.png"), dpi=150)
plt.show()

## 4. Chart 3: Average Sales With Promo vs Without Promo
Evaluate the impact of promotions (`promo == 1` vs `promo == 0`) on average store sales.

In [ ]:
# Compare promotional vs non-promotional days
labels = ["No Promo (0)", "Promo (1)"]
promo_means = [df[df["promo"] == 0]["sales"].mean(), df[df["promo"] == 1]["sales"].mean()]

plt.figure(figsize=(6, 5))
bars = plt.bar(labels, promo_means, color=["#ff7f0e", "#1f77b4"], edgecolor="black", alpha=0.85, width=0.5)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 100, f"{int(height):,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.title("Average Daily Sales: Promo vs No Promo", fontsize=14, pad=12)
plt.xlabel("Promotion Status", fontsize=12)
plt.ylabel("Average Sales per Store", fontsize=12)
plt.ylim(0, 9500)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

# Save chart
plt.savefig(os.path.join(fig_dir, "sales_promo_effect.png"), dpi=150)
plt.show()

## 5. Chart 4: Average Sales per City
Compare store sales performance across the 5 mapped German cities.

In [ ]:
# Average sales per city
city_order = ["Berlin", "Hamburg", "Munich", "Cologne", "Frankfurt"]
avg_city = df.groupby("city")["sales"].mean().reindex(city_order)

plt.figure(figsize=(8, 5))
bars = plt.bar(avg_city.index, avg_city.values, color="#9467bd", edgecolor="black", alpha=0.85, width=0.55)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 100, f"{int(height):,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.title("Average Daily Sales per City", fontsize=14, pad=12)
plt.xlabel("City", fontsize=12)
plt.ylabel("Average Sales per Store", fontsize=12)
plt.ylim(0, 8500)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

# Save chart
plt.savefig(os.path.join(fig_dir, "sales_per_city.png"), dpi=150)
plt.show()

## 6. Baseline Model: Sales-Only Linear Regression
We build a simple baseline regression model using only past sales and calendar features:
- `sales_lag7`: Sales from 7 days ago (matching the exact day of the week last week)
- `dayofweek`: Integer day of the week (0 = Monday, ..., 6 = Sunday)
- `promo`: Whether a promotional campaign was active (0 or 1)

Target variable: `sales`

**Why build lag7 by DATE and not with shift(7)?**
Because retail stores are closed on Sundays and public holidays, `shift(7)` would look back 7 open business days rather than 7 calendar days. Merging on `date + 7 days` guarantees we look at the exact same day of the week from the previous week.

In [ ]:
# 1. Read data/processed/sales_clean.csv with parse_dates=["date"]
sales_df = pd.read_csv(data_path, parse_dates=["date"])

# 2. Build sales_lag7 by DATE (not with shift(7))
# Copy the table, add 7 days to the date column, and rename sales to sales_lag7
df_lag = sales_df[["store_id", "date", "sales"]].copy()
df_lag["date"] = df_lag["date"] + pd.Timedelta(days=7)
df_lag = df_lag.rename(columns={"sales": "sales_lag7"})

# Left merge back on store_id and date
model_df = sales_df.merge(df_lag, on=["store_id", "date"], how="left")

# Drop rows with no sales_lag7 (first week and days following store closures)
model_df = model_df.dropna(subset=["sales_lag7"]).copy()

# 3. Add dayofweek from the date (0 = Monday, 6 = Sunday)
model_df["dayofweek"] = model_df["date"].dt.dayofweek

print(f"Total modeling rows with lag7: {len(model_df):,}")
model_df[["date", "store_id", "sales", "sales_lag7", "dayofweek", "promo"]].head()

### Chronological Train / Test Split
We split by date: train on all historical data up to the last 60 days, and test on the last 60 days without random shuffling.

In [ ]:
# 4. Define features and target
features = ["sales_lag7", "dayofweek", "promo"]
target = "sales"

# 5. Split by date: train on older data, test on the last 60 days
max_date = model_df["date"].max()
split_date = max_date - pd.Timedelta(days=60)

train_df = model_df[model_df["date"] < split_date].copy()
test_df = model_df[model_df["date"] >= split_date].copy()

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print("Train set:")
print(f"  Row count:  {len(train_df):,}")
print(f"  Date range: {train_df['date'].min().strftime('%Y-%m-%d')} to {train_df['date'].max().strftime('%Y-%m-%d')}")
print()
print("Test set (last 60 days):")
print(f"  Row count:  {len(test_df):,}")
print(f"  Date range: {test_df['date'].min().strftime('%Y-%m-%d')} to {test_df['date'].max().strftime('%Y-%m-%d')}")

### Train Baseline Linear Regression & Save Metrics
We train the model, compute Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) on the test set, and save the result to `results/baseline_metrics.csv`.

In [ ]:
# 6. Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Calculate MAE and RMSE
mae = mean_absolute_error(y_test, y_pred)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))

print("=== Baseline Model Performance (Test Set) ===")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print()
print("Model Coefficients:")
for feat, coef in zip(features, model.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  Intercept: {model.intercept_:.4f}")

# Save metrics to results/baseline_metrics.csv
results_dir = "results" if os.path.exists("results") else "../results"
metrics_file = os.path.join(results_dir, "baseline_metrics.csv")
metrics_df = pd.DataFrame([{
    "model": "baseline_sales_only",
    "MAE": round(mae, 2),
    "RMSE": round(rmse, 2)
}])
metrics_df.to_csv(metrics_file, index=False)
print()
print(f"Saved metrics to {metrics_file}:")
print(metrics_df)

### 7. Explanation of MAE for This Data
**What does the MAE number mean?**  
- The MAE of **1,116.93** means that, on average across the 50 stores during the 60-day test period, our baseline prediction misses actual daily store sales by about **1,117 units** (or euros).  
- In the test period, average daily sales per store is approximately **7,000 units**. An MAE of ~1,117 represents an average error of roughly **16%**, providing a simple and reasonable baseline for Person 2 (weather) and Person 3 (social trends) to improve upon.